In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
n = 200

first_names = [
    "Aarav", "Aanya", "Aditya", "Anaya", "Arjun", "Diya", "Ishaan",
    "Isha", "Kabir", "Kavya", "Meera", "Neha", "Rohan", "Saanvi",
    "Sahil", "Tara", "Vivaan", "Zoya", "Reyansh", "Myra"
]
last_names = [
    "Sharma", "Verma", "Singh", "Gupta", "Joshi", "Rawat", "Kapoor",
    "Mehta", "Rana", "Pant", "Malik", "Das", "Khan", "Nair"
]

patient_ids = np.arange(1001, 1001 + n)
names = [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n)]
ages = rng.integers(1, 90, size=n)
genders = rng.choice(["M", "F"], size=n)

# Fictional fields for an ML demonstration.
bmi = np.round(np.clip(rng.normal(24.5, 4.8, n), 15, 42), 1)
systolic_bp = np.clip(np.round(rng.normal(122, 17, n)), 85, 195).astype(int)
glucose = np.clip(np.round(rng.normal(105, 25, n)), 65, 240).astype(int)
smoker = rng.choice([0, 1], size=n, p=[0.78, 0.22])

# Artificial target rule for teaching only, not a medical standard.
risk_score = (
    (ages >= 55).astype(int)
    + (bmi >= 30).astype(int)
    + (systolic_bp >= 140).astype(int)
    + (glucose >= 140).astype(int)
    + smoker
)
risk_label = (risk_score >= 2).astype(int)

ml_df = pd.DataFrame({
    "patient_id": patient_ids, "name": names, "age": ages, "gender": genders,
    "bmi": bmi, "systolic_bp": systolic_bp, "glucose": glucose,
    "smoker": smoker, "risk_label": risk_label
})
postgres_df = ml_df[["patient_id", "name", "age", "gender"]].copy()

print("Generated patient records:", len(ml_df))
display(ml_df.head())

Generated patient records: 200


,patient_id,name,age,gender,bmi,systolic_bp,glucose,smoker,risk_label
0,1001,Aanya Malik,89,F,26.6,102,106,0,0
1,1002,Saanvi Kapoor,70,F,27.0,108,100,1,1
2,1003,Kabir Khan,29,F,25.8,126,78,0,0
3,1004,Aanya Pant,87,M,17.7,149,101,1,1
4,1005,Arjun Verma,45,F,15.0,101,86,0,0


In [3]:
print("Dataset shape:", ml_df.shape)
print("\nMissing values per column:")
display(ml_df.isna().sum().to_frame("missing_values"))

print("\nLabel distribution (0 = simulated lower risk, 1 = simulated higher risk):")
display(ml_df["risk_label"].value_counts().sort_index().to_frame("count"))

Dataset shape: (200, 9)

Missing values per column:


,missing_values
patient_id,0
name,0
age,0
gender,0
bmi,0
systolic_bp,0
glucose,0
smoker,0
risk_label,0



Label distribution (0 = simulated lower risk, 1 = simulated higher risk):


,count
risk_label,
0,155
1,45


In [4]:
postgres_df.to_csv("smartcare_patients_200.csv", index=False)
ml_df.to_csv("smartcare_synthetic_ml_200.csv", index=False)

print("Saved smartcare_patients_200.csv")
print("Saved smartcare_synthetic_ml_200.csv")

Saved smartcare_patients_200.csv
Saved smartcare_synthetic_ml_200.csv


In [7]:
feature_cols = ["age", "bmi", "systolic_bp", "glucose", "smoker"]
X = ml_df[feature_cols]
y = ml_df["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 80.00%

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.88        31
           1       0.57      0.44      0.50         9

    accuracy                           0.80        40
   macro avg       0.71      0.67      0.69        40
weighted avg       0.79      0.80      0.79        40

Confusion matrix:
[[28  3]
 [ 5  4]]


In [10]:
assert len(postgres_df) == 200
assert postgres_df["patient_id"].is_unique
assert postgres_df[["patient_id", "name", "age", "gender"]].notna().all().all()

print("Checks passed: 200 rows, unique IDs, and no missing base fields.")
print("fictional data only. This model is not for medical decisions.")

Checks passed: 200 rows, unique IDs, and no missing base fields.
fictional data only. This model is not for medical decisions.
